In [19]:
''' 
This document demonstrates how to use LangChain with a PDF document.

'''

from langchain_core.prompts import PromptTemplate

'''Docling has a better answer so far'''
from langchain_docling import DoclingLoader

import getpass
import os


if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for GROQ_API_KEY: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("llama3-70b-8192", model_provider="groq")
# llm = init_chat_model("gpt-4o-mini", model_provider="openai")


In [2]:
def get_pdf_content(file_path):
    loader = DoclingLoader(file_path)
    docs = []
    for doc in loader.lazy_load():
        docs.append(doc)
    return "".join(doc.page_content for doc in docs)

In [3]:
import chromadb

persistent_client = chromadb.PersistentClient()
try:
    collection = persistent_client.get_collection(name="MOSFETs")
    print(f"Collection found.")
except:
    collection = None
    print(f"Collection not found. Please create one.")


Collection found.


In [21]:
document = 'tms320f28069m-q1'
# document = 'LAUNCHXL-F28069M'
# document = 'TI_csd19538q2'
# device_part_number = 'Infineon_IMZC120R017M2H'
# device_part_number = 'Wolfspeed_C3M0016120K'

if not collection:
    # Create a new collection
    collection = persistent_client.create_collection(name="MOSFETs")
    print(f"Collection {document} created.")
# Add documents to the collection
doc_data = collection.get(ids=[document])
if not doc_data['ids']:
    file_path = f"./docs/{document}.pdf"
    docs_content = get_pdf_content(file_path)
    collection.add(
        documents=[docs_content],
        metadatas=[{"source": file_path}],
        ids=[document],
    )
# print(mosfet_data["documents"])
docs_content = doc_data["documents"]

In [17]:
limit   = 500    # tune this to balance request‑size vs. memory
offset  = 0
results = collection.get(
    limit=limit,
    offset=offset,
)
print(f"Found {len(results['documents'])} documents.")

Found 3 documents.


In [20]:
question = "How can I connect I measure a differential sensing?"

template = """Use the following pieces of context to answer the question at the end.
{context}
Question: {question}
If the question is not answerable based on the context, please say "I don't know".
"""
custom_rag_prompt = PromptTemplate.from_template(template)

prompt = custom_rag_prompt.invoke({"context": docs_content, "question": question})
answer = llm.invoke(prompt)
print(f"response: {answer.content}") 


APIStatusError: Error code: 413 - {'error': {'message': 'Request too large for model `llama3-70b-8192` in organization `org_01jrv8enwjekc9gm5nnghtqrrn` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Requested 140306, please reduce your message size and try again. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

## To get answer in structured format, use the following code

In [ ]:

from pydantic import BaseModel, Field
class ResponseFormatter(BaseModel):
    """Always use this tool to structure your response to the user."""
    min: str = Field(description="value for the minimum")
    max: str = Field(description="value for the maximum")
    typical: str = Field(description="value for the typical")
    unit: str = Field(description="Unit of the value")
    temperature: str = Field(description="Temperature of the value")
llm=llm.bind_tools([ResponseFormatter])

ai_msg = llm.invoke(f"get the threshold values from {answer.content} and return them in the format of the ResponseFormatter tool")

pydantic_object = ResponseFormatter.model_validate(ai_msg.tool_calls[0]["args"])

print(f"{document} vth sepcs are:\n {pydantic_object}")    